# 2stage-CPLS compression of the clean AIS

Run this notebook live (interactive figures) on Binder: [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/smehri/CATraj/main?labpath=notebooks%2F03_two_stage_cpls_compression.ipynb)  
Or view it on [nbviewer](https://nbviewer.org/github/smehri/CATraj/blob/main/notebooks/03_two_stage_cpls_compression.ipynb). The figures are interactive (hvplot / bokeh: zoom, pan, hover, click a legend entry), so they do not show on github.com, whose notebook viewer does not run JavaScript.

Trajectory compression keeps the points of a track that matter and drops the
rest. This notebook compresses **all** the clean AIS of the chosen dataset with
**2stage-CPLS**, the two-stage topology-preserving method of

> Mehri, S., Alesheikh, A. A., Basiri, A. (2021). *A Contextual Hybrid Model for Vessel Movement Prediction.* IEEE Access 9, 45600–45613. [doi:10.1109/ACCESS.2021.3066463](https://doi.org/10.1109/ACCESS.2021.3066463)

and saves the result for the prediction notebook
(`05_clstm_prediction.ipynb`), which trains the CLSTM networks on it.

1. **The method**: CPLS and 2stage-CPLS, what each stage does
2. **Load** the clean AIS and the coastline
3. **Trips**: the clean tracks are split into trips at gaps; tiny trips are dropped
4. **Compress**: `two_stage_cpls_compress` over every trip, with the coastline as barrier
5. **Save**: `compressed_ais.parquet`, `trips_ais.parquet` and a summary table
6. **Results**: compression ratio overall and by vessel type, vessels before and after on a map
7. **Checks**: no compressed segment crosses the coastline where the original did not

**Inputs**: `AIS/ais.parquet` and `Coastline/coastline.parquet` of the preprocessed
folder (written by `01_preprocessing.ipynb`). **Outputs**: `results/03_two_stage_cpls_compression/<DATA>/`.
The package is built on [MovingPandas](https://movingpandas.org):

> Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54–68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)

## Data

The notebook compresses the preprocessed AIS (`ais.parquet`: cleaned, median-filtered
positions of every vessel, with speed, course, status and vessel type) and uses the
preprocessed coastline polygons (`coastline.parquet`) as barriers.

By default it runs on the **committed, preprocessed two-day sample** (2015-11-09 to
2015-11-11 UTC, Brest area, 565 vessels). To process the **full archive**, run
`notebooks/01_preprocessing.ipynb` first (its download and preprocessing sections), then set
`DATA = "full"` below, or run the notebook with the environment variable `CATRAJ_DATA=full`.
The full archive covers October to December 2015 and February to March 2016. **January 2016
is left out of every dataset**: the Zenodo archive has no sea state for it (its
`oc_january.csv` is a byte copy of `oc_november.csv`), so those AIS positions could not be
annotated with the ocean context and are not used anywhere.

Data: Ray, C. et al. (2019), *Heterogeneous integrated dataset for Maritime Intelligence,
Surveillance, and Reconnaissance*, Data in Brief 25, 104141,
[doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141); Zenodo
[doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595). AIS from NARI, Brest
(CC BY-NC-SA 4.0: anything derived from it, including this notebook's outputs, is shared
under the same licence); coastline from the European Environment Agency (acknowledge the source).

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")   # see catraj/__init__.py

DATA = os.environ.get("CATRAJ_DATA", "sample")      # "sample" or "full"
N_JOBS = int(os.environ.get("CATRAJ_N_JOBS", "-1"))  # -1: every core; 1: serial

import time
from pathlib import Path

import geopandas as gpd
import geoviews as gv
import holoviews as hv
import hvplot.pandas  # noqa: F401  -- registers .hvplot
import numpy as np
import pandas as pd
import shapely
from pyproj import Transformer

hv.extension("bokeh")                  # interactive figures: zoom, pan, hover, click a legend entry

from catraj import parallel_map
from catraj.compression import (Barriers, compression_ratio, segment_crossings,
                                two_stage_cpls_compress, two_stage_cpls_select)
from catraj.datasets import load_ais, load_coastline, repo_root, sample_root
from catraj.preprocessing import split_trips

if DATA not in ("sample", "full"):
    raise ValueError(f"DATA must be 'sample' or 'full', not {DATA!r}")
ROOT = repo_root()
PRE = sample_root() if DATA == "sample" else ROOT / "data" / "zenodo_1167595" / "preprocessed"
OUT = ROOT / "results" / "03_two_stage_cpls_compression" / DATA
OUT.mkdir(parents=True, exist_ok=True)

# ---- the parameters of the method (README / CPLS_PHASE1_GUIDE.md) ---------------
EPSILON = 1_000.0        # Douglas-Peucker threshold, metres
SPEED_EPSILON = 1.54     # speed threshold, metres per second (3 knots)
STAY_RADIUS = 100.0      # a movement stay: within 100 m ...
STAY_DURATION = 600.0    # ... for at least 600 s
STATUS_COLUMN = "navigational_status"   # codes 1 (at anchor) and 5 (moored) are stays
# ---- the trips --------------------------------------------------------------------
MAX_GAP = 1_800.0        # a gap of over 30 minutes between two reports starts a new trip
MIN_TRIP_POINTS = 10     # trips with fewer positions are dropped

print(f"DATA = {DATA!r}, N_JOBS = {N_JOBS}\ninput:  {PRE}\noutput: {OUT}")

DATA = 'full', N_JOBS = -1
input:  C:\dev\CATraj\data\zenodo_1167595\preprocessed
output: C:\dev\CATraj\results\03_two_stage_cpls_compression\full


## Parallel processing

Three steps of this notebook run in parallel, all through the package's `n_jobs`
(`catraj.parallel_map` / `map_groups`, `src/catraj/_parallel.py`):

* **`split_trips(..., n_jobs=N_JOBS)`**: every vessel's track is split independently of
  the others, so the vessels are shared out over the processes.
* **`two_stage_cpls_compress(..., by="trip_id", n_jobs=N_JOBS)`**: every trip is
  compressed independently of the others (the speed series, the stays and the topological
  checks only ever look at one trip), so the trips are shared out. They are grouped
  into chunks of about equal number of positions, because the cost depends on a trip's length.
* **The checks** at the end (`catraj.parallel_map` over the trips): the same, one task per trip.

`N_JOBS` follows the scikit-learn convention: `1` runs everything serially in this
process, `-1` uses every core, `k` uses `k` processes. The results come back in task order and
every trip is computed by the same code either way, so **parallel and serial runs give
identical results**; only the wall time changes. Each worker receives its share of the
trips (pickled) and the coastline (about 1 MB), so on the full archive the memory in use is
roughly the data frame in this process plus the chunks in flight in the workers. To run
serially, set `N_JOBS = 1` or start Jupyter with `CATRAJ_N_JOBS=1`.

## 1 · The method

**CPLS** (Context-aware Piecewise Linear Segmentation, Algorithm 1 of the paper) is
Douglas–Peucker with a memory of the world. Douglas–Peucker (DP) drops a point when it lies
within ε of the line between its neighbours: a statement about *distance* only. On a
coastal route it will replace a bend around a headland by a chord over the land, or move an
island from one side of the track to the other. CPLS runs DP and then, **inside every
segment that DP collapsed**, removes the points one by one, front to back, and puts a
point back whenever removing it would change the track's *topological signature*
(after Tienaah et al.): the relation of the simplified track with each neighbouring
barrier (DE-9IM: no crossing of land that the original did not cross), the side on which each
barrier lies (a nine-cell direction model), and how the track meets itself (no new
self-intersection, none lost, no fold-back of two consecutive segments). The result is always a
superset of the DP result and never introduces a new crossing of the coastline.
`help(catraj.compression.cpls)` lists the decisions behind it.

**Why a second stage.** Plain CPLS on raw AIS keeps most points of a vessel at a berth:
the position of a moored vessel jitters by a few metres, its track crosses itself thousands
of times, and every crossing is part of the signature. Worse, simplified away, "it appears
as if the vessel moves slowly, whereas in the original uncompressed trajectory the vessel
stops moving for a period of time" (de Vries and van Someren, 2012).
**2stage-CPLS** (Algorithm 2 of the paper) therefore works in stages:

1. **Stays.** A *stay point* is a position whose AIS status says at anchor or moored (codes
   1 and 5), or that belongs to a *movement stay* (Li et al.'s stay-point detection: the
   vessel remains within `STAY_RADIUS` of a spot for at least `STAY_DURATION`). Each stay is
   replaced by two vertices at its centroid, the arrival and the departure, with their own
   times: the jitter is gone, the duration of the stop is kept.
2. **Speed.** Douglas–Peucker on the *speed series* (speed from positions and time, with error
   `|v - v'|`, `v'` interpolated in time) keeps the points where the speed changes by more
   than `SPEED_EPSILON`: a vessel that brakes, accelerates or stops keeps the points that say so.
3. **Position.** Between those forced vertices, Douglas–Peucker with `EPSILON` metres, and
   then the topological pass of CPLS over the collapsed track.

The compressed track is therefore the stay-collapsed track, simplified; the
guarantee about the coastline is relative to that collapsed track (see the checks at the end).
`help(catraj.compression.two_stage)` lists every decision, with the sentence of the paper behind it.

**What we use.** `EPSILON = 1,000 m`, `SPEED_EPSILON = 1.54 m/s`, stay radius 100 m, stay
duration 600 s and the status column `navigational_status`: the parameters of the README.
All five CPLS relations stay on. The coastline is the barrier.

**Filtering.** `two_stage_cpls_compress` median-filters the positions over 30 s by default
(and the speed series with the same window). The preprocessed AIS **already is** filtered
(notebook 01 applies the same 30 s median), so filtering again would only smooth smooth
data, change the positions the CLSTM later trains on and cost time. We pass
`filter_window=None`: the compressed rows are then exactly rows of the clean data, with
their original values, except that a stay's two kept points move to the stay's centroid.

## 2 · Load the clean AIS and the barriers

The AIS is a `GeoDataFrame` in EPSG:32630 (UTM zone 30N, metres, as the distance thresholds
need), sorted by `(mmsi, unix_time)`. The barriers are the coastline's land polygons; `Barriers(...)`
indexes them once so that every trip reuses the index.

In [2]:
t0 = time.perf_counter()
ais = load_ais(PRE)
coast = load_coastline(PRE)
barriers = Barriers(coast)
LOAD_S = time.perf_counter() - t0
N_ALL = len(ais)
print(f"{N_ALL:,} positions, {ais.mmsi.nunique():,} vessels, "
      f"{pd.to_datetime(ais.unix_time.min(), unit='s'):%Y-%m-%d} to "
      f"{pd.to_datetime(ais.unix_time.max(), unit='s'):%Y-%m-%d} UTC; "
      f"{len(coast):,} land polygons ({shapely.get_num_coordinates(coast.geometry.values).sum():,} vertices); "
      f"loaded in {LOAD_S:.1f} s")
ais[["mmsi", "unix_time", "x", "y", "speed_over_ground", "navigational_status", "vessel_type"]].head()

14,906,535 positions, 4,720 vessels, 2015-10-01 to 2016-03-31 UTC; 875 land polygons (66,149 vertices); loaded in 4.7 s


,mmsi,unix_time,x,y,speed_over_ground,navigational_status,vessel_type
0,205067000,1445550804,567611.461228,5.554564e+06,12.95,0.0,Unknown
1,205067000,1445550818,567611.461228,5.554564e+06,12.95,0.0,Unknown
2,205067000,1445556790,531318.813270,5.543685e+06,11.40,0.0,Unknown
3,205067000,1445557411,527780.793575,5.542699e+06,11.30,0.0,Unknown
4,205067000,1445557417,527769.532057,5.542696e+06,11.30,0.0,Unknown


## 3 · Trips

A vessel's AIS history is one long track: over days or months it is many voyages joined
end to end, with periods out of range in between. Compression and, later, the prediction
network work on **trips**, so `catraj.preprocessing.split_trips` cuts each vessel's track
wherever two consecutive reports are **more than `MAX_GAP` = 30 minutes** apart: the
vessel was out of range or its transponder was off, so the positions on either side are
not one movement and a chord across the gap would invent a path. 30 minutes is far longer than
the reporting interval of AIS (seconds to a few minutes) and short enough not to join separate voyages.

Stays are *not* used to cut the trips here: a stay is part of a trip (the vessel waits and
leaves again), and finding and collapsing stays is exactly what stage 1 of 2stage-CPLS does
inside each trip. So `split_trips` is called with no stay rule (`stay_radius=None`).

Some pieces are tiny: a handful of reports before the vessel goes out of range, a single
fix between two long gaps. They carry no trajectory a model could learn from, so trips with fewer than
`MIN_TRIP_POINTS` = 10 positions are **dropped** from both the uncompressed and the
compressed data. Every trip gets a `trip_id`, unique over the whole dataset, which
both output files carry. The uncompressed, trip-split data is saved as `trips_ais.parquet`:
notebook 05 trains on it too, so the trip definition is the same for both models.

In [3]:
t0 = time.perf_counter()
ais = split_trips(ais, by="mmsi", time_column="unix_time", max_gap=MAX_GAP,
                  stay_radius=None, stay_duration=None, stay_column=None,
                  trip_column="trip", n_jobs=N_JOBS)
SPLIT_S = time.perf_counter() - t0

sizes = ais.groupby(["mmsi", "trip"], sort=False).size()
small = sizes < MIN_TRIP_POINTS
N_TRIPS_ALL, N_TINY, N_TINY_POINTS = len(sizes), int(small.sum()), int(sizes[small].sum())
big = sizes[~small].index.to_frame(index=False)
trips = ais.merge(big, on=["mmsi", "trip"])
trips = trips.sort_values(["mmsi", "unix_time"], kind="stable").reset_index(drop=True)
trips["trip_id"] = trips.groupby(["mmsi", "trip"], sort=False).ngroup().astype("int64")
trips = trips.drop(columns="trip")
del ais
N = len(trips)
N_TRIPS = trips.trip_id.nunique()
print(f"split into {N_TRIPS_ALL:,} trips at gaps of over {MAX_GAP / 60:.0f} min "
      f"({SPLIT_S:.1f} s on {N_JOBS} job(s))")
print(f"dropped {N_TINY:,} tiny trips of fewer than {MIN_TRIP_POINTS} positions "
      f"({N_TINY_POINTS:,} positions, {N_TINY_POINTS / N_ALL:.2%} of the data)")
print(f"kept {N_TRIPS:,} trips, {N:,} positions, {trips.mmsi.nunique():,} vessels")
tl = trips.groupby("trip_id").size()
print("positions per kept trip:", tl.describe().round(0).astype(int).to_dict())

split into 76,058 trips at gaps of over 30 min (26.8 s on -1 job(s))
dropped 56,651 tiny trips of fewer than 10 positions (129,117 positions, 0.87% of the data)
kept 19,407 trips, 14,777,418 positions, 2,076 vessels
positions per kept trip: {'count': 19407, 'mean': 761, 'std': 4883, 'min': 10, '25%': 19, '50%': 48, '75%': 313, 'max': 192484}


## 4 · Compress

`two_stage_cpls_compress(traj, epsilon, speed_epsilon, barriers, ...)` takes the points of
all trips as one `GeoDataFrame`, with `by="trip_id"` compressing every trip on its own, and
returns only the retained rows, every column intact (index label, time, speed, course, status,
vessel type). Its `stay` column is True on the two kept points of every stay.

The vessels are compressed type by type, so that each vessel type has its own timing in the
summary; within one type the trips are shared out over `N_JOBS` processes. Seconds are wall time.

In [4]:
PARAMS = dict(time_column="unix_time", by="trip_id", status_column=STATUS_COLUMN,
              stay_radius=STAY_RADIUS, stay_duration=STAY_DURATION, stay_column="stay",
              filter_window=None, n_jobs=N_JOBS)       # the data is already filtered (section 1)

kept_parts, rows = [], []
T0 = time.perf_counter()
for vtype, part in trips.groupby("vessel_type", sort=False):
    t0 = time.perf_counter()
    res = two_stage_cpls_compress(part, EPSILON, SPEED_EPSILON, barriers, **PARAMS)
    seconds = time.perf_counter() - t0
    kept_parts.append(res)
    rows.append({"vessel_type": vtype, "vessels": part.mmsi.nunique(),
                 "trips": part.trip_id.nunique(), "points_in": len(part),
                 "points_out": len(res), "stay_points_out": int(res.stay.sum()),
                 "compression_ratio_pct": compression_ratio(len(res), len(part)),
                 "seconds": seconds})
COMPRESS_S = time.perf_counter() - T0
compressed = pd.concat(kept_parts).sort_index()        # the index is trips' row number: input order
print(f"compressed {N:,} -> {len(compressed):,} points in {COMPRESS_S:.1f} s "
      f"on {N_JOBS} job(s)")

compressed 14,777,418 -> 1,636,505 points in 261.2 s on -1 job(s)


## 5 · Save

The compressed rows keep the geometry of the retained points, in which a stay's two ends are
at the stay's centroid. The `x`, `y`, `lon`, `lat` columns are rewritten from that geometry
so that they agree with it (for every other row they are unchanged). Written under
`results/03_two_stage_cpls_compression/<DATA>/`:

* `compressed_ais.parquet`: the compressed points with the columns the CLSTM needs
  (`mmsi`, `trip_id`, `unix_time`, `x`, `y`, `lon`, `lat`, `speed_over_ground`,
  `course_over_ground`, `vessel_type`, `navigational_status`, `stay`) plus `true_heading`,
  `rate_of_turn` and `ship_type_code`;
* `trips_ais.parquet`: the same columns (without `stay`) of the **uncompressed** clean AIS,
  split into trips in the same way (same `trip_id`);
* `summary.csv`: points in and out, compression ratio and seconds, per vessel type and in total.

In [5]:
to_ll = Transformer.from_crs(32630, 4326, always_xy=True)
compressed = compressed.copy()
gx, gy = compressed.geometry.x.to_numpy(), compressed.geometry.y.to_numpy()
moved = (gx != compressed.x.to_numpy()) | (gy != compressed.y.to_numpy())
lon, lat = to_ll.transform(gx[moved], gy[moved])
compressed.loc[moved, "lon"], compressed.loc[moved, "lat"] = lon, lat
compressed["x"], compressed["y"] = gx, gy
print(f"{int(moved.sum()):,} rows moved to a stay centroid; stay column True on "
      f"{int(compressed.stay.sum()):,} rows")

COLUMNS = ["mmsi", "trip_id", "unix_time", "x", "y", "lon", "lat", "speed_over_ground",
           "course_over_ground", "vessel_type", "navigational_status", "true_heading",
           "rate_of_turn", "ship_type_code"]
pd.DataFrame(compressed[COLUMNS + ["stay"]]).to_parquet(OUT / "compressed_ais.parquet", index=False)
pd.DataFrame(trips[COLUMNS]).to_parquet(OUT / "trips_ais.parquet", index=False)

summary = pd.DataFrame(rows).sort_values("points_in", ascending=False)
total = {"vessel_type": "ALL", "vessels": trips.mmsi.nunique(), "trips": N_TRIPS,
         "points_in": N, "points_out": len(compressed),
         "stay_points_out": int(compressed.stay.sum()),
         "compression_ratio_pct": compression_ratio(len(compressed), N),
         "seconds": COMPRESS_S}
summary = pd.concat([summary, pd.DataFrame([total])], ignore_index=True)
summary["split_trips_seconds"] = SPLIT_S      # the same for every row: one pass over all vessels
summary["n_jobs"] = N_JOBS
summary.to_csv(OUT / "summary.csv", index=False)
for f in sorted(OUT.iterdir()):
    print(f"{f.name:24s} {f.stat().st_size / 1e6:8.2f} MB")

21,468 rows moved to a stay centroid; stay column True on 21,470 rows


compressed_ais.parquet      67.49 MB
summary.csv                  0.00 MB
trips_ais.parquet          436.11 MB


## 6 · Results

**Compression ratio** R is the percentage of points removed (Eq. 9 of the CADP paper):
`R = (1 - points out / points in) * 100`. First overall and by vessel type; then
the vessel types are compared in a chart. Different vessels compress very differently:
a vessel that sails a straight line or sits at a berth gets a very high ratio, one that
turns and changes speed all the time keeps many points.

In [6]:
show = summary.copy()
show["compression_ratio_pct"] = show.compression_ratio_pct.round(2)
show["seconds"] = show.seconds.round(1)
print(f"overall: {N:,} -> {len(compressed):,} points, R = {compression_ratio(len(compressed), N):.2f} %, "
      f"{N / len(compressed):.1f} points in per point kept")
show.drop(columns=["split_trips_seconds", "n_jobs"]).set_index("vessel_type")

overall: 14,777,418 -> 1,636,505 points, R = 88.93 %, 9.0 points in per point kept


,vessels,trips,points_in,points_out,stay_points_out,compression_ratio_pct,seconds
vessel_type,,,,,,,
Passenger,38,1311,4045030,442929,4988,89.05,35.5
Fishing,262,7339,2866090,445859,5746,84.44,88.5
Unknown,235,1701,2189901,204878,2752,90.64,25.1
Cargo,904,4082,1252446,239527,507,80.88,23.8
Tug,30,156,1213339,15824,880,98.70,6.9
Tanker,297,1038,626062,69512,318,88.90,10.3
Other,54,765,594229,40666,960,93.16,11.6
Military Ops,12,103,540227,20158,162,96.27,8.1
Pilot Vessel,5,445,308202,66132,466,78.54,11.9


In [7]:
by_type = summary[summary.vessel_type != "ALL"].sort_values("compression_ratio_pct").copy()
R_ALL = compression_ratio(len(compressed), N)
by_type["label"] = [f"{r:.1f} %  ({n:,} pts)" for r, n in zip(by_type.compression_ratio_pct, by_type.points_in)]
by_type["pos"] = by_type.compression_ratio_pct + 1.0
bars = by_type.hvplot.barh(x="vessel_type", y="compression_ratio_pct", color="#3b6fb6", xlabel="",
                           ylabel="compression ratio R (% of points removed)",
                           hover_cols=["points_in", "points_out", "vessels", "trips"], ylim=(0, 112),
                           width=800, height=int(26 * len(by_type) + 110),
                           title="2stage-CPLS: compression ratio by vessel type")
labels = hv.Labels(by_type, ["vessel_type", "pos"], "label").opts(text_align="left", text_font_size="8pt")
bars * labels * hv.VLine(R_ALL).opts(color="#cf222e", line_width=1.5)

:Overlay
   .Bars.I   :Bars   [vessel_type]   (compression_ratio_pct,points_in,points_out,vessels,trips)
   .Labels.I :Labels   [vessel_type,pos]   (label)
   .VLine.I  :VLine   [x,y]

### Vessels before and after

Three vessels, one from each of the three vessel types with most positions (the vessel with
the most positions up to 5,000 that stays inside the area of the coastline and has a stay): the clean positions in grey (trip by trip), the compressed points joined by
their segments in blue, the **stays** (the two centroid points of every stay) as orange squares,
the coastline as land polygons. The left panel is the original, the right panel the compressed track. Both panels are interactive: zoom and pan, hover a point for its time and speed.


In [8]:
LAND, INK, ORANGE, BLUE = "#e3dcc0", "#0d1117", "#e16f24", "#1f5fa8"
coast_ll = coast.to_crs(4326)
coast_ll["geometry"] = coast_ll.geometry.simplify(0.0005)        # ~50 m: plenty for a map, much lighter


def pick_vessels(k=3, cap=5_000):
    """From each of the k largest vessel types: the vessel with most positions (up to the cap)
    that stays inside the coastline's extent and has a stay, so that land and stays are in view."""
    bx0, by0, bx1, by1 = coast.total_bounds
    box = trips.groupby("mmsi").agg(x0=("x", "min"), x1=("x", "max"), y0=("y", "min"), y1=("y", "max"),
                                    n=("x", "size"), vtype=("vessel_type", "first"))
    inside = box[(box.x0 > bx0) & (box.x1 < bx1) & (box.y0 > by0) & (box.y1 < by1) & (box.n <= cap)]
    with_stay = inside[inside.index.isin(compressed.mmsi[compressed.stay])]
    out = []
    for vtype in summary[summary.vessel_type != "ALL"].vessel_type.head(k):
        for pool in (with_stay, inside, box[box.n <= cap]):
            pool = pool[pool.vtype == vtype]
            if len(pool):
                out.append((vtype, int(pool.n.idxmax()))); break
    return out


def basemap(lon0, lon1, lat0, lat1):
    """Map tiles with the land polygons of the coastline inside the box, drawn first (below the tracks)."""
    base = gv.tile_sources.CartoLight()
    box = gpd.GeoDataFrame(geometry=[shapely.box(lon0, lat0, lon1, lat1)], crs=4326)
    part = gpd.clip(coast_ll, box)
    if len(part):
        base = base * part.hvplot.polygons(geo=True, color=LAND, line_color="#8a7f5a", line_width=0.5,
                                           alpha=0.6, hover=False)
    return base


def trip_lines(df, **opts):
    """One line per trip (no chord across the gap between two trips), as a geographic path layer."""
    g = gpd.GeoDataFrame({"trip_id": df.trip_id.unique()},
                         geometry=[shapely.LineString(np.c_[t.lon, t.lat]) for _, t in df.groupby("trip_id", sort=False)],
                         crs=4326)
    return g.hvplot.paths(geo=True, hover=False, **opts)


def with_time(df):
    df = df[["trip_id", "lon", "lat", "unix_time", "speed_over_ground"]].copy()
    df["time"] = pd.to_datetime(df.unix_time, unit="s")
    return df


HOVER = ["time", "speed_over_ground"]
chosen = pick_vessels()
panels = []
for vtype, mmsi in chosen:
    before = with_time(trips[trips.mmsi == mmsi])
    aft = compressed[compressed.mmsi == mmsi]
    after = with_time(aft)
    st = with_time(aft[aft.stay])
    pad = max(0.02, 0.12 * max(np.ptp(before.lon) * 0.7, np.ptp(before.lat)))
    base = basemap(before.lon.min() - pad, before.lon.max() + pad, before.lat.min() - pad, before.lat.max() + pad)
    m0 = base
    m0 = m0 * trip_lines(before, color="#777777", line_width=1)
    m0 = m0 * before.hvplot.points(x="lon", y="lat", geo=True, color="#555555", size=3, hover_cols=HOVER)
    m1 = base
    m1 = m1 * trip_lines(after, color=BLUE, line_width=2)
    m1 = m1 * after.hvplot.points(x="lon", y="lat", geo=True, color=INK, size=6, hover_cols=HOVER)
    if len(st):
        m1 = m1 * st.hvplot.points(x="lon", y="lat", geo=True, color=ORANGE, marker="square", size=60,
                                   line_color=INK, hover_cols=["time"], label="stay (centroid)")
    m0 = m0.opts(width=520, height=440, title=f"{vtype}, MMSI {mmsi}: original, {len(before):,} points")
    m1 = m1.opts(width=520, height=440,
                 title=f"compressed, {len(after):,} points (R = {compression_ratio(len(after), len(before)):.1f} %)")
    panels += [m0, m1]
hv.Layout(panels).cols(2).opts(shared_axes=False)

:Layout
   .Overlay.I   :Overlay
      .WMTS.I     :WMTS   [Longitude,Latitude]
      .Polygons.I :Polygons   [Longitude,Latitude]
      .Path.I     :Path   [Longitude,Latitude]
      .Points.I   :Points   [lon,lat]   (time,speed_over_ground)
   .Overlay.II  :Overlay
      .WMTS.I                                                  :WMTS   [Longitude,Latitude]
      .Polygons.I                                              :Polygons   [Longitude,Latitude]
      .Path.I                                                  :Path   [Longitude,Latitude]
      .Points.I                                                :Points   [lon,lat]   (time,speed_over_ground)
      .Points.Stay_left_parenthesis_centroid_right_parenthesis :Points   [lon,lat]   (time)
   .Overlay.III :Overlay
      .WMTS.I     :WMTS   [Longitude,Latitude]
      .Polygons.I :Polygons   [Longitude,Latitude]
      .Path.I     :Path   [Longitude,Latitude]
      .Points.I   :Points   [lon,lat]   (time,speed_over_ground)
   .Overlay.IV  :Overlay
      .WMTS.I                                                  :WMTS   [Longitude,Latitude]
      .Polygons.I                                              :Polygons   [Longitude,Latitude]
      .Path.I                                                  :Path   [Longitude,Latitude]
      .Points.I                                                :Points   [lon,lat]   (time,speed_over_ground)
      .Points.Stay_left_parenthesis_centroid_right_parenthesis :Points   [lon,lat]   (time)
   .Overlay.V   :Overlay
      .WMTS.I     :WMTS   [Longitude,Latitude]
      .Polygons.I :Polygons   [Longitude,Latitude]
      .Path.I     :Path   [Longitude,Latitude]
      .Points.I   :Points   [lon,lat]   (time,speed_over_ground)
   .Overlay.VI  :Overlay
      .WMTS.I                                                  :WMTS   [Longitude,Latitude]
      .Polygons.I                                              :Polygons   [Longitude,Latitude]
      .Path.I                                                  :Path   [Longitude,Latitude]
      .Points.I                                                :Points   [lon,lat]   (time,speed_over_ground)
      .Points.Stay_left_parenthesis_centroid_right_parenthesis :Points   [lon,lat]   (time)

### Stays

A stay is replaced by two points at its centroid, the arrival and the departure, with their own times, so the
position stays in one place while the time passes. The map shows the compressed points of the
dataset (a random sample of at most 20,000 drawn, with up to 6,000 of them stay points): ordinary points in blue, stay points in orange, over the coastline.
Stays sit where vessels wait: in ports, at moorings and at anchor, along the coast. The line below counts
the stay points kept (each stay of two or more points leaves two flagged points, a one-point stay one).

In [9]:
print(f"{int(compressed.stay.sum()):,} stay points kept; "
      f"{compressed[compressed.stay].groupby('trip_id').ngroups:,} trips contain a stay; "
      f"{compressed[compressed.stay].mmsi.nunique():,} vessels")
st_all = compressed[compressed.stay]
st_draw = st_all.sample(min(6_000, len(st_all)), random_state=0)
ns_all = compressed[~compressed.stay]
ns_draw = ns_all.sample(min(20_000 - len(st_draw), len(ns_all)), random_state=0)
cols = ["lon", "lat", "vessel_type", "speed_over_ground"]
coast_view = coast_ll.copy()
coast_view["geometry"] = coast_view.geometry.simplify(0.002)      # ~200 m: the whole area at once
land_all = (gv.tile_sources.CartoLight()
            * coast_view.hvplot.polygons(geo=True, color=LAND, line_color="#8a7f5a", line_width=0.4, alpha=0.6, hover=False))
(land_all
 * pd.DataFrame(ns_draw[cols]).hvplot.points(x="lon", y="lat", geo=True, color=BLUE, size=3, alpha=0.5,
                                             hover_cols=["vessel_type", "speed_over_ground"], label="compressed points")
 * pd.DataFrame(st_draw[cols]).hvplot.points(x="lon", y="lat", geo=True, color=ORANGE, size=7,
                                             hover_cols=["vessel_type", "speed_over_ground"], label="stay points (centroids)")
 ).opts(width=900, height=700, legend_position="top_right",
        title=f"Compressed points and stays (a sample of {len(ns_draw) + len(st_draw):,} of {len(compressed):,})")

21,470 stay points kept; 6,211 trips contain a stay; 599 vessels


:Overlay
   .WMTS.I                                                          :WMTS   [Longitude,Latitude]
   .Polygons.I                                                      :Polygons   [Longitude,Latitude]
   .Points.Compressed_points                                        :Points   [lon,lat]   (vessel_type,speed_over_ground)
   .Points.Stay_points_left_parenthesis_centroids_right_parenthesis :Points   [lon,lat]   (vessel_type,speed_over_ground)

## 7 · Checks

**No new crossing of the coastline.** The package defines a *new crossing* as a segment of the
compressed track that touches a barrier although the stretch of the original track it replaces does not
(`segment_crossings(x, y, barriers, keep=mask, new_only=True)`). Two measurements:

* **Against the raw track**, for every trip: the clean positions and the retained
  ones. This is not what 2stage-CPLS guarantees, because a stay's points are moved to their
  centroid, up to the extent of the stay away from where the vessel was, so a segment that starts or ends
  at a moved stay point can touch land where the raw one did not (a vessel at a quay, whose position is within metres
  of the coastline polygon). The count says how many of the new crossings, if any, are at a stay point.
* **Against the collapsed track**, the reference the algorithm keeps: `two_stage_cpls_select` is run again
  on a random sample of trips (up to 2,000, all of them on the sample), and its own
  collapsed coordinates and mask are measured. The guarantee says this count is **zero**; the cell also
  checks that the recomputed mask is exactly the set of rows saved.

Both run in parallel over the trips.

In [10]:
def new_crossings(args):
    """New crossings of one trip against the raw track; also how many touch a stay point."""
    x, y, keep, stay = args
    seg = segment_crossings(x, y, barriers, keep=keep, new_only=True)
    if not seg.any():
        return 0, 0
    kept = np.flatnonzero(keep)
    at_stay = stay[kept[:-1]] | stay[kept[1:]]
    return int(seg.sum()), int((seg & at_stay).sum())


t0 = time.perf_counter()
keep_all = np.zeros(N, dtype=bool)
keep_all[compressed.index.to_numpy()] = True
stay_all = np.zeros(N, dtype=bool)
stay_all[compressed.index[compressed.stay.to_numpy()].to_numpy()] = True
xs, ys = trips.x.to_numpy(), trips.y.to_numpy()
bounds = np.flatnonzero(np.diff(trips.trip_id.to_numpy())) + 1
starts, ends = np.r_[0, bounds], np.r_[bounds, N]
tasks = [(xs[lo:hi], ys[lo:hi], keep_all[lo:hi], stay_all[lo:hi]) for lo, hi in zip(starts, ends)]
sizes_ = (ends - starts).tolist()
res = np.array(parallel_map(new_crossings, tasks, N_JOBS, sizes=sizes_))
print(f"against the raw track: {int(res[:, 0].sum())} new crossing(s) in {int((res[:, 0] > 0).sum())} "
      f"of {len(tasks):,} trips ({int(res[:, 1].sum())} of them at a segment end that is a stay point); "
      f"{time.perf_counter() - t0:.1f} s")

against the raw track: 76 new crossing(s) in 76 of 19,407 trips (76 of them at a segment end that is a stay point); 6.7 s


In [11]:
t0 = time.perf_counter()
rng = np.random.default_rng(0)
n_check = min(2_000, len(tasks))
which = np.sort(rng.choice(len(tasks), n_check, replace=False))
status_all = trips.navigational_status.to_numpy(dtype=float)
unix_all = trips.unix_time.to_numpy(dtype=float)


def collapsed_crossings(i):
    lo, hi = starts[i], ends[i]
    sel = two_stage_cpls_select(xs[lo:hi], ys[lo:hi], unix_all[lo:hi], EPSILON, SPEED_EPSILON,
                                barriers, status=status_all[lo:hi], stay_radius=STAY_RADIUS,
                                stay_duration=STAY_DURATION, speed_window=None)
    new = int(segment_crossings(sel.x, sel.y, barriers, keep=sel.mask, new_only=True).sum())
    same = bool((sel.mask == keep_all[lo:hi]).all())
    return new, int(same)


chk = np.array(parallel_map(collapsed_crossings, list(which), N_JOBS,
                            sizes=[sizes_[i] for i in which]))
print(f"against the collapsed track: {int(chk[:, 0].sum())} new crossing(s) in {n_check:,} trips "
      f"checked; the recomputed selection equals the saved rows in {int(chk[:, 1].sum()):,} of them; "
      f"{time.perf_counter() - t0:.1f} s")
assert chk[:, 0].sum() == 0 and chk[:, 1].all(), "2stage-CPLS guarantee violated"

against the collapsed track: 0 new crossing(s) in 2,000 trips checked; the recomputed selection equals the saved rows in 2,000 of them; 148.5 s


**Reading the checks.** Against the collapsed track there is no new crossing, as guaranteed,
and recomputing the selection with the array function reproduces the saved rows exactly (the
notebook stops with an error otherwise). Against the raw track, the first line counts what the
move of stays to their centroid costs, if anything: the documented price of replacing the
jitter of a stay by its centroid.

## 8 · Timings and files

All times are wall time of this run.

In [12]:
print(f"load {LOAD_S:.1f} s | split_trips {SPLIT_S:.1f} s | compress {COMPRESS_S:.1f} s "
      f"| N_JOBS = {N_JOBS} ({os.cpu_count()} logical cores)")
print(f"{N / COMPRESS_S:,.0f} positions per second compressed")
pd.DataFrame({"file": [f.name for f in sorted(OUT.iterdir())],
              "MB": [round(f.stat().st_size / 1e6, 2) for f in sorted(OUT.iterdir())]}).set_index("file")

load 4.7 s | split_trips 26.8 s | compress 261.2 s | N_JOBS = -1 (16 logical cores)
56,572 positions per second compressed


,MB
file,
compressed_ais.parquet,67.49
summary.csv,0.00
trips_ais.parquet,436.11


## Notes

* The compressed data is **a row subset** of `trips_ais.parquet` (same `trip_id`, same
  `unix_time`, same measured columns); the only changed values are the position of the stay
  points (`x`, `y`, `lon`, `lat`), which sit at their stay's centroid.
* A trip needs at least 10 positions to be kept; both outputs use the same trips.
* The speed threshold is in metres per second for UNIX-second times: 1.54 m/s is 3 knots.
* Everything generated here is written under `results/` and is not committed; the committed
  copy of this notebook is the executed run on the sample. The run on the full archive is saved as
  `notebooks/full_runs/03_two_stage_cpls_compression.ipynb`.

**Citing.** MovingPandas: Graser, A. (2019), GI_Forum 7(1), 54–68,
[doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54). CPLS and
2stage-CPLS: Mehri, S., Alesheikh, A. A., Basiri, A. (2021), IEEE Access 9, 45600–45613,
[doi:10.1109/ACCESS.2021.3066463](https://doi.org/10.1109/ACCESS.2021.3066463). Stay points: Li, Q.
et al. (2008), ACM SIGSPATIAL GIS; Yu, Y. (2015). Two-stage compression: de Vries, G. K. D. and van
Someren, M. (2012), Expert Systems with Applications 39(18), 13426–13439. Data: Ray, C. et al. (2019),
Data in Brief 25, 104141, [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141); Zenodo
[doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595). Coastline: European Environment Agency,
"EEA coastline for analysis" (2013). Anything derived from the AIS data falls under CC BY-NC-SA 4.0.